In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import time

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_API = "https://www.cbioportal.org/api"

STUDIES = {
    "BRCA": "brca_tcga",
    "LUAD": "luad_tcga",
    "KIRC": "kirc_tcga",
    "GBM":  "gbm_tcga",
    "COAD": "coad_tcga"
}

HEADERS = {"Accept": "application/json"}

In [2]:
# Hallmark cancer genes: oncogenes, tumor suppressors, RTKs, transcription factors
CANCER_GENES = [
    "TP53","KRAS","EGFR","BRCA1","BRCA2","PIK3CA","PTEN","APC","VHL","MYC",
    "CDKN2A","RB1","BRAF","NRAS","HRAS","ERBB2","MET","ALK","RET","FGFR1",
    "FGFR2","FGFR3","IDH1","IDH2","TERT","ARID1A","SMAD4","FBXW7","NOTCH1",
    "CDH1","RUNX1","FLT3","NPM1","DNMT3A","TET2","ASXL1","SF3B1","U2AF1",
    "SRSF2","STAG2","RAD21","SMC1A","SMC3","CTNNB1","AXIN1","AXIN2","TCF7L2",
    "LEF1","WNT5A","FZD1","DVL1","GSK3B","CCND1","CCND2","CCND3","CDK4","CDK6",
    "CDK2","CDKN1A","CDKN1B","CDKN2B","E2F1","E2F3","PCNA","MCM2","MCM7",
    "AURKA","AURKB","PLK1","BUB1","MAD2L1","BIRC5","BCL2","BCL2L1","MCL1",
    "BAX","BAK1","BID","PUMA","NOXA","CASP3","CASP8","CASP9","APAF1","CYCS",
    "XIAP","DIABLO","HIF1A","VEGFA","VEGFC","FLT1","KDR","PDGFRA","PDGFRB",
    "KIT","CSFR1","FLT3","JAK1","JAK2","STAT3","STAT5A","STAT5B","SOCS1",
    "SOCS3","PIK3R1","PIK3R2","AKT1","AKT2","AKT3","MTOR","TSC1","TSC2",
    "PHLPP1","PHLPP2","FOXO1","FOXO3","GSK3A","RPTOR","RICTOR","DEPTOR",
    "RAF1","MAP2K1","MAP2K2","MAPK1","MAPK3","DUSP4","DUSP6","SPRY2","SPRY4",
    "RASA1","NF1","RASSF1","RASSF2","DAPK1","LATS1","LATS2","YAP1","WWTR1",
    "TEAD1","TEAD4","MOB1A","SAV1","STK3","STK4","PTCH1","SMO","GLI1","GLI2",
    "HHIP","SHH","DHH","IHH","SUFU","BOC","CDO","GAS1","FOXM1","FOXC2",
    "SNAI1","SNAI2","TWIST1","TWIST2","ZEB1","ZEB2","VIM","CDH2","FN1",
    "MMP2","MMP9","MMP14","TIMP1","TIMP2","ITGB1","ITGB3","ITGA5","ITGAV",
    "FAK1","SRC","YES1","FYN","LCK","ZAP70","SYK","BTK","ITK","TEC",
    "RHOA","RHOC","RAC1","CDC42","PAK1","PAK2","ROCK1","ROCK2","LIMK1",
    "COFILIN","ARPC2","WASF1","DIAPH1","VASP","ENAH","EZH2","SUZ12","EED",
    "KMT2A","KMT2C","KMT2D","SETD2","KDM5C","KDM6A","CREBBP","EP300","HDAC1",
    "HDAC2","SIRT1","BRD4","BRD2","MED12","MED13","CDK8","CDK19","CCNC",
    "SMARCA4","SMARCA2","SMARCB1","ARID2","PBRM1","ARID1B","BCL6","MYB",
    "GATA1","GATA2","GATA3","PAX5","EBF1","IKZF1","IRF4","MUM1","POU2AF1",
    "SOX2","SOX9","SOX17","OCT4","NANOG","KLF4","NKX2-1","FOXA1","FOXA2",
    "GATA4","GATA6","HNF1A","HNF4A","CDX2","MSX1","DLL3","DLL4","JAG1",
    "JAG2","NOTCH2","NOTCH3","NOTCH4","HEY1","HEY2","HES1","HES5","RBPJ",
    "MAML1","NUMB","FBXW7","ITCH","EGFL7","ANGPT1","ANGPT2","TIE2","NRP1",
    "EPHA2","EPHB4","EFNA1","EFNB2","ROBO1","SLIT2","DCC","UNC5C","NETRIN1",
    "PLCG1","PLCG2","IP3R1","CAMK2A","CALM1","CAMK4","PKC","PRKCA","PRKCB",
    "PRKCZ","DAG1","GNA11","GNAQ","GNA12","GNA13","RGNEF","TRIO","PREX1",
    "TIAM1","DOCK1","DOCK2","VAV1","VAV2","VAV3","ABL1","ABL2","BCR","PML",
    "RARA","PML","NUP98","DEK","NUP214","CBFB","RUNX1T1","ETV6","PDGFRB",
    "SSX1","SSX2","SYT","TLE1","ASPSCR1","TFE3","PRCC","NONO","SFPQ","PRDM1",
    "IRF1","IRF2","IRF3","IRF7","MAVS","STING1","CGAS","TBK1","IKBKE",
    "DDX58","IFIH1","TRIM63","TRIM25","TRIM27","NLRP3","CASP1","IL1B","IL18",
    "GSDMD","PYCARD","NLRC4","AIM2","ZBP1","RIPK1","RIPK3","MLKL","FADD",
    "TRADD","TRAF2","TRAF5","BIRC2","BIRC3","SHARPIN","LUBAC","OTULIN",
    "CYLD","A20","LUBAC","NEMO","IKBKB","IKBKA","NFKB1","NFKB2","RELA",
    "RELB","REL","NFKBIA","NFKBIB","BCAP31","ERAD","ATF6","IRE1","PERK",
    "GRP78","CHOP","ATF4","XBP1","EDEM1","SEL1L","DERL1","VCP","UBQLN2",
    "P62","NBR1","OPTN","NDP52","TAX1BP1","ATG5","ATG7","ATG12","BECN1",
    "PIK3C3","UVRAG","RUBCN","AMBRA1","ATG14","ATG16L1","LC3B","LAMP1",
    "RAB7A","SNX","ESCRT","TSG101","VPS4A","CHMP4B","PDCD6IP"
]

print(f"Gene panel size: {len(CANCER_GENES)} genes")

Gene panel size: 417 genes


CompletedProcess(args=['pip', 'install', 'pyarrow'], returncode=0)

In [8]:
import subprocess
subprocess.run(["pip", "install", "pyarrow"], check=True)

CompletedProcess(args=['pip', 'install', 'pyarrow'], returncode=0)

In [5]:
r = requests.get(
    f"{BASE_API}/studies/coad_tcga/samples?pageSize=200",
    headers=HEADERS
)
print("Status:", r.status_code)
print("Response type:", type(r.json()))
print("Raw (first 300 chars):", str(r.json())[:300])

Status: 404
Response type: <class 'dict'>
Raw (first 300 chars): {'message': 'Study not found: coad_tcga'}


In [6]:
STUDIES["COAD"]       = "coadread_tcga"
RNA_PROFILES["COAD"]  = "coadread_tcga_rna_seq_v2_mrna"

cache = RAW_DIR / "COAD_expression.parquet"
if cache.exists():
    cache.unlink()  # delete the failed cache if any

m = fetch_expression_fixed("COAD", RNA_PROFILES["COAD"], entrez_ids, symbol_map)
m.to_parquet(cache)
matrices["COAD"] = m

[COAD] 200 samples | profile: coadread_tcga_rna_seq_v2_mrna
  [COAD] genes 300–383 done...
[COAD] ✅ Matrix: 383 genes × 62 samples


In [7]:
import time
import requests
import pandas as pd
from pathlib import Path

BASE_API = "https://www.cbioportal.org/api"
HEADERS  = {"Accept": "application/json"}
RAW_DIR  = Path("../data/raw")

# ── 1. Resolve hugo symbols → entrez IDs once ──────────────────────────────
print("Resolving gene symbols to Entrez IDs...")
r = requests.post(
    f"{BASE_API}/genes/fetch?geneIdType=HUGO_GENE_SYMBOL",
    json=CANCER_GENES,
    headers={**HEADERS, "Content-Type": "application/json"}
)
gene_info   = r.json()
symbol_map  = {g["entrezGeneId"]: g["hugoGeneSymbol"] for g in gene_info}
entrez_ids  = list(symbol_map.keys())
print(f"Resolved {len(entrez_ids)} / {len(CANCER_GENES)} genes\n")

# ── 2. Per-study RNA-seq profile (rna_seq_v2_mrna) ─────────────────────────
RNA_PROFILES = {
    "BRCA": "brca_tcga_rna_seq_v2_mrna",
    "LUAD": "luad_tcga_rna_seq_v2_mrna",
    "KIRC": "kirc_tcga_rna_seq_v2_mrna",
    "GBM":  "gbm_tcga_mrna",               # GBM has no RNA-seq v2, falls back to microarray
    "COAD": "coad_tcga_rna_seq_v2_mrna",
}

# ── 3. Fetch function ───────────────────────────────────────────────────────
def fetch_expression(label, profile_id, entrez_ids, symbol_map, n_samples=200):
    study_id = STUDIES[label]

    # Get samples
    r = requests.get(
        f"{BASE_API}/studies/{study_id}/samples?pageSize={n_samples}",
        headers=HEADERS
    )
    samples = [s["sampleId"] for s in r.json()]
    print(f"[{label}] {len(samples)} samples | profile: {profile_id}")

    # Fetch in batches of 100 genes
    all_records = []
    batch_size  = 100

    for i in range(0, len(entrez_ids), batch_size):
        batch = entrez_ids[i : i + batch_size]
        r = requests.post(
            f"{BASE_API}/molecular-profiles/{profile_id}/molecular-data/fetch?projection=SUMMARY",
            json={"entrezGeneIds": batch, "sampleIds": samples},
            headers={**HEADERS, "Content-Type": "application/json"}
        )
        if r.status_code == 200:
            all_records.extend(r.json())
        else:
            print(f"  ⚠️  batch {i} returned {r.status_code}: {r.text[:100]}")
        time.sleep(0.3)
        print(f"  [{label}] genes {i}–{i+len(batch)} done...", end="\r")

    # Build genes × samples matrix
    df     = pd.DataFrame(all_records)[["entrezGeneId", "sampleId", "value"]]
    df["gene"] = df["entrezGeneId"].map(symbol_map)
    matrix = df.pivot_table(index="gene", columns="sampleId", values="value")
    matrix.columns.name = None
    print(f"\n[{label}] ✅ Matrix: {matrix.shape[0]} genes × {matrix.shape[1]} samples")
    return matrix

# ── 4. Run all studies (with parquet cache) ────────────────────────────────
matrices = {}
for label in STUDIES:
    cache = RAW_DIR / f"{label}_expression.parquet"
    if cache.exists():
        print(f"[{label}] Loading from cache...")
        matrices[label] = pd.read_parquet(cache)
    else:
        m = fetch_expression(label, RNA_PROFILES[label], entrez_ids, symbol_map)
        m.to_parquet(cache)
        matrices[label] = m

Resolving gene symbols to Entrez IDs...
Resolved 383 / 417 genes

[BRCA] Loading from cache...
[LUAD] Loading from cache...
[KIRC] Loading from cache...
[GBM] Loading from cache...
[COAD] Loading from cache...


In [8]:
for label, mat in matrices.items():
    print(f"{label}: {mat.shape[1]} samples × {mat.shape[0]} genes | NaN%: {mat.isna().mean().mean()*100:.1f}%")

BRCA: 199 samples × 383 genes | NaN%: 0.0%
LUAD: 133 samples × 383 genes | NaN%: 0.0%
KIRC: 198 samples × 383 genes | NaN%: 0.0%
GBM: 86 samples × 378 genes | NaN%: 0.0%
COAD: 62 samples × 383 genes | NaN%: 0.0%


In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

BASE_DIR = Path(r"C:\Projects\cancer-classifier")
RAW_DIR  = BASE_DIR / "data" / "raw"
BASE_API = "https://www.cbioportal.org/api"
HEADERS  = {"Accept": "application/json"}

# Load the gene mapping we built earlier
symbol_map = {g["entrezGeneId"]: g["hugoGeneSymbol"] for g in requests.post(
    f"{BASE_API}/genes/fetch?geneIdType=HUGO_GENE_SYMBOL",
    json=pd.read_csv(BASE_DIR / "data" / "processed" / "selected_genes.csv").iloc[:, 0].tolist(),
    headers={**HEADERS, "Content-Type": "application/json"}
).json()}
entrez_ids = list(symbol_map.keys())
print(f"Loaded {len(entrez_ids)} genes")

# Verify LGG RNA-seq profile exists
r = requests.get(f"{BASE_API}/studies/lgg_tcga/molecular-profiles", headers=HEADERS)
profiles = [p["molecularProfileId"] for p in r.json()]
print("LGG profiles:", [p for p in profiles if "rna" in p.lower()])

Loaded 340 genes
LGG profiles: ['lgg_tcga_mrna', 'lgg_tcga_mrna_median_Zscores', 'lgg_tcga_mrna_median_all_sample_Zscores', 'lgg_tcga_rna_seq_v2_mrna', 'lgg_tcga_rna_seq_v2_mrna_median_Zscores', 'lgg_tcga_rna_seq_v2_mrna_median_all_sample_Zscores']


In [2]:
def fetch_samples_safe(study_id, n_samples, retries=3):
    for attempt in range(retries):
        r = requests.get(
            f"{BASE_API}/studies/{study_id}/samples?pageSize={n_samples}",
            headers=HEADERS
        )
        data = r.json()
        if isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):
            return [s["sampleId"] for s in data]
        print(f"  ⚠️ Bad response (attempt {attempt+1}): {str(data)[:100]}")
        time.sleep(2)
    raise ValueError(f"Failed after {retries} attempts")

def fetch_expression_fixed(label, profile_id, study_id, entrez_ids, symbol_map, n_samples=200):
    samples = fetch_samples_safe(study_id, n_samples)
    print(f"[{label}] {len(samples)} samples | profile: {profile_id}")

    all_records = []
    for i in range(0, len(entrez_ids), 100):
        batch = entrez_ids[i : i + 100]
        r = requests.post(
            f"{BASE_API}/molecular-profiles/{profile_id}/molecular-data/fetch?projection=SUMMARY",
            json={"entrezGeneIds": batch, "sampleIds": samples},
            headers={**HEADERS, "Content-Type": "application/json"}
        )
        if r.status_code == 200:
            all_records.extend(r.json())
        else:
            print(f"  ⚠️ batch {i} → {r.status_code}")
        time.sleep(0.3)
        print(f"  [{label}] genes {i}–{i+len(batch)} done...", end="\r")

    df     = pd.DataFrame(all_records)[["entrezGeneId", "sampleId", "value"]]
    df["gene"] = df["entrezGeneId"].map(symbol_map)
    matrix = df.pivot_table(index="gene", columns="sampleId", values="value")
    matrix.columns.name = None
    print(f"\n[{label}] ✅ Matrix: {matrix.shape[0]} genes × {matrix.shape[1]} samples")
    return matrix

# Delete old GBM cache and fetch LGG instead
old_gbm = RAW_DIR / "GBM_expression.parquet"
if old_gbm.exists():
    old_gbm.unlink()
    print("🗑️  Deleted old GBM cache")

lgg_cache = RAW_DIR / "LGG_expression.parquet"
if not lgg_cache.exists():
    m = fetch_expression_fixed(
        "LGG", "lgg_tcga_rna_seq_v2_mrna", "lgg_tcga",
        entrez_ids, symbol_map, n_samples=200
    )
    m.to_parquet(lgg_cache)
else:
    print("[LGG] Loading from cache...")

print("✅ LGG ready!")

🗑️  Deleted old GBM cache
[LGG] 200 samples | profile: lgg_tcga_rna_seq_v2_mrna
  [LGG] genes 300–340 done...
[LGG] ✅ Matrix: 340 genes × 200 samples
✅ LGG ready!
